# W17-D1 实验：基线升级裁决 477→499 × 五族归因 × 账本式 vs 地毯式基线

> 配套阅读材料：`第17周-Day1-W39Digest与基线升级裁决477到499及g0x并轨立案.md`（md=叙事，本件=可执行验证）。
> 核心概念：**基线不是墙，是账本**——升级的合法性来自程序（取证 RED → 归因 → 裁决 → 重冻结记债 → 复验 GREEN），
> 而不是数字对上。五个实验全部读真实文件 / 真实 git 历史，断言失败即实验失败。

In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)

## 实验 1：477→499 增量复算 + 五族 × 迁移归因（读真实文件）

增量不是抄 md：直接 diff 两份冻结基线 + 主仓清单；归因不是人工判断：直接读四份 `.up.sql` 的 `CREATE TABLE`。

In [ ]:
import hashlib, re
from pathlib import Path

def table_set(p):
    lines = Path(p).read_text().splitlines()
    return {l.strip() for l in lines if l.strip() and not l.startswith("#")}

W39 = table_set("/root/learning-notebooks/semantic-model/governance/canonical-baseline-w39.txt")
W40 = table_set("/root/learning-notebooks/semantic-model/governance/canonical-baseline-w40.txt")
REPO = table_set("/root/lnkcre/backend/internal/platform/database/testdata/canonical_tables.txt")

fp = lambda s: hashlib.sha256("\n".join(sorted(s)).encode()).hexdigest()[:16]
print(f"w39 基线 {len(W39)} 表，指纹 {fp(W39)}")
print(f"w40 基线 {len(W40)} 表，指纹 {fp(W40)}")
print(f"主仓现实 {len(REPO)} 表，指纹 {fp(REPO)}")
added   = W40 - W39
removed = W39 - W40
print(f"\n增量 +{len(added)} / -{len(removed)}")
assert len(W39) == 477 and len(W40) == 499 and len(REPO) == 499
assert len(added) == 22 and not removed
assert W40 == REPO, "w40 冻结件必须与主仓清单逐表一致（对齐源裁决）"
assert fp(W40) == "7995cb4839ddc5b1", "集合指纹与 digest 记录一致"
print("断言全过：477→499、+22/-0、w40=主仓、指纹 7995cb4839ddc5b1")

In [ ]:
# 归因：四份迁移 .up.sql 直读 CREATE TABLE（双目录并集原则）
MIG_DIR = Path("/root/lnkcre/backend/internal/platform/database/migrations-pg")
migration_tables = {}
for mid in ["000227", "000229", "000233", "000234"]:
    f = next(MIG_DIR.glob(f"{mid}_*.up.sql"))
    tables = set(re.findall(r"CREATE TABLE (?:IF NOT EXISTS )?([a-z_]+)", f.read_text()))
    migration_tables[mid] = tables
    print(f"{mid} {f.name}: {len(tables)} 表 -> {sorted(tables)}")

attributed = set().union(*migration_tables.values())
print(f"\n四迁移归因合计 {len(attributed)} 表")
assert attributed == added, "迁移归因必须精确覆盖 22 表增量（不多不少）"
print("断言过：22 表全部有出生证明（迁移级归因闭环）")

In [ ]:
# 五族分类：按表名语义分族，与 digest §5 的五族口径互相验证
def family(t):
    if t.startswith("indicator_monthly_target") or t == "target_indicators": return "indicator-target"
    if t.startswith("leasing_progress") or t.startswith("leasing_stage"):    return "leasing-progress"
    if t.startswith("unit_leasing") or t == "leasing_plan_recalc_batches":  return "unit_leasing"
    if t.startswith("leasing_polic"):                                       return "leasing-policy"
    if t.startswith("unit_pricing_batch") or t == "price_authority_constraints": return "unit-pricing"
    return "?"

from collections import Counter
fam_count = Counter(family(t) for t in added)
fam_mig = {}   # 族 -> {迁移: 表数}
for t in added:
    fam_mig.setdefault(family(t), Counter())
    for mid, ts in migration_tables.items():
        if t in ts: fam_mig[family(t)][mid] += 1
for fam in ["indicator-target", "leasing-progress", "unit_leasing", "leasing-policy", "unit-pricing"]:
    print(f"{fam:16s} {fam_count[fam]:2d} 表  {dict(fam_mig[fam])}")
assert dict(fam_count) == {"indicator-target": 3, "leasing-progress": 9, "unit_leasing": 3,
                           "leasing-policy": 3, "unit-pricing": 4}
print("\n断言过：五族 3/9/3/3/4 = 22，与 digest/提案登记完全一致")

In [ ]:
# 可视化：五族 × 迁移归因矩阵（堆叠条形图）
fams = ["indicator-target", "leasing-progress", "unit_leasing", "leasing-policy", "unit-pricing"]
migs = ["000227", "000229", "000233", "000234"]
colors = {"000227": "#4C72B0", "000229": "#DD8452", "000233": "#55A868", "000234": "#C44E52"}
fig, ax = plt.subplots(figsize=(9, 4.5))
bottom = [0] * len(fams)
for m in migs:
    vals = [fam_mig[f].get(m, 0) for f in fams]
    ax.bar(fams, vals, bottom=bottom, label=f"迁移 {m}", color=colors[m])
    bottom = [b + v for b, v in zip(bottom, vals)]
for i, f in enumerate(fams):
    ax.text(i, fam_count[f] + 0.25, str(fam_count[f]), ha="center", fontweight="bold")
ax.set_ylabel("表数"); ax.set_title("canonical 477→499：五族 × 迁移归因（22 表全部有出生证明）")
ax.legend(title="来源迁移", fontsize=9)
plt.tight_layout(); plt.savefig("/root/learning-notebooks/w17d1_family_attribution.png", dpi=110)
plt.show()
print("图存 w17d1_family_attribution.png")

## 实验 2：集合指纹为什么必须「排序归一」

门的基线指纹是对**表集合**（排序后拼接）做 sha256，不是对文件原文——本实验证明这一设计选择：
文件行序无关（任何人重排清单不改指纹），但集合内容一变指纹必变。

In [ ]:
import random
random.seed(42)
lines_w40 = sorted(W40)
naive = lambda lines: hashlib.sha256("\n".join(lines).encode()).hexdigest()[:16]

# ① 排序归一指纹：打乱行序 100 次，指纹恒定
stable = all(fp(random.sample(lines_w40, len(lines_w40))) == fp(W40) for _ in range(100))
# ② 朴素原文指纹：仅交换两行，指纹即变
swapped = lines_w40[:]; swapped[0], swapped[1] = swapped[1], swapped[0]
naive_fragile = naive(swapped) != naive(lines_w40)
# ③ 内容敏感性：加一张假表，排序指纹必变
sensitive = fp(W40 | {"__fake_table__"}) != fp(W40)
print(f"排序归一指纹 行序无关性（100 次打乱恒定）: {stable}")
print(f"朴素原文指纹   交换两行即变            : {naive_fragile}")
print(f"排序归一指纹   +1 假表必变            : {sensitive}")
assert stable and naive_fragile and sensitive
print("断言过：指纹 = 内容的函数，不是排版/顺序的函数——归一化是集合门的数学地基")

## 实验 3：账本式基线 vs 地毯式基线（本日核心概念）

同一场「+22 表」事件、同一句「基线升级到 499」，两种策略走三条后续事件，看分叉在哪。
地毯式：只记 count，数字对上就 GREEN。账本式：记集合 + 逐表裁决状态（citing）。

In [ ]:
class CarpetBaseline:
    """地毯式：只存表数。升级=把数字改成新的；GREEN=数字对上。"""
    def __init__(self, tables): self.count = len(tables)
    def upgrade(self, tables): self.count = len(tables)          # 直接覆盖数字
    def observe(self, tables):
        return "GREEN" if len(tables) == self.count else "RED"   # 只能比数字

class LedgerBaseline:
    """账本式：存集合 + 逐表 citing 状态。升级=重冻结并记 carrying_debt。"""
    def __init__(self, tables): self.tables = set(tables); self.citing = {}; self.debt_events = []
    def upgrade(self, tables, adjudicated):
        new = set(tables) - self.tables
        self.citing.update({t: adjudicated for t in new})         # 裁决过的入账
        uncited = {t for t in new if not adjudicated}
        if uncited: self.debt_events.append({"+uncited": len(uncited)})  # carrying-debt 显式入账
        self.tables = set(tables)
    def observe(self, tables, open_changes=frozenset()):
        cur = set(tables)
        added, removed = cur - self.tables, self.tables - cur
        cited = {t for t in added if t in open_changes}
        leak = added - cited
        verdict = []
        if added and leak: verdict.append(f"BROKEN(+{len(added)} added/{len(leak)} leak)")
        if removed:        verdict.append(f"BROKEN(-{len(removed)} 死登记)")
        if added and not leak: verdict.append(f"OK(+{len(added)} 全 cited)")
        return " | ".join(verdict) if verdict else "GREEN"

# --- 事件序列回放 ---
universe0 = set(f"t{i:03d}" for i in range(477))                  # w39 世界
ev1_added = set(f"new_{i:02d}" for i in range(22))                # 09-18~21 活事件：+22
ev2      = ({"new_99"}, {"t000"})                                 # 未来事件：+1 新表 -1 旧表（净 0）
ev3      = ({"new_100"}, set())                                   # 未来事件：+1 但带 citing

carpet = CarpetBaseline(universe0); ledger = LedgerBaseline(universe0)
after1 = universe0 | ev1_added
carpet.upgrade(after1); ledger.upgrade(after1, adjudicated=False)  # 今日裁决：升级但 citing 未清

print("事件1（+22 后升级）:")
print(f"  地毯式 observe: {carpet.observe(after1)}   （数字对上，安静）")
print(f"  账本式 observe: {ledger.observe(after1)}   carrying_debt={ledger.debt_events}")

u2 = (after1 - ev2[1]) | ev2[0]
print("\n事件2（+1 新表 -1 旧表，净 0）:")
print(f"  地毯式 observe: {carpet.observe(u2)}   ← 净零变化完全隐身")
print(f"  账本式 observe: {ledger.observe(u2)}")

u3 = u2 | ev3[0]
print("\n事件3（再 +1 新表；漂移被看见后，new_99/new_100 都补了 citing）:")
print(f"  地毯式 observe: {carpet.observe(u3)}   <- 只知道数字多了 1，不知道是什么/为什么")
print(f"  账本式 observe: {ledger.observe(u3, open_changes=ev2[0] | ev3[0])}")

assert carpet.observe(u2) == "GREEN"          # 地毯式对净零漂移失明（假绿）
assert "leak" in ledger.observe(u2) and "死登记" in ledger.observe(u2)
v3 = ledger.observe(u3, open_changes=ev2[0] | ev3[0])
assert "全 cited" in v3 and "死登记" in v3    # citing 补齐治 leak；死登记方向要另案裁决
print("\n断言过：计数守不住边界（假绿），集合+逐表裁决才守得住——这就是 G-07 把 S1 项从『计数比对』升级为『逐表集合判决』的原因")

In [ ]:
# 可视化：两策略在三条事件上的可见性对比
events = ["+22后升级", "+1-1(净0)", "+1带citing"]
carpet_visible = [0, 0, 0]                       # 地毯式：事件1升级后绿、事件2假绿、事件3仍盲
ledger_visible = [22, 2, 1]                      # 账本式：分别可见 22 / 2（1 leak+1 死登记）/ 1（cited 留痕）
x = range(len(events)); w = 0.36
fig, ax = plt.subplots(figsize=(8.5, 4.2))
b1 = ax.bar([i - w/2 for i in x], carpet_visible, w, label="地毯式（计数）", color="#A5A5A5")
b2 = ax.bar([i + w/2 for i in x], ledger_visible, w, label="账本式（集合+逐表裁决）", color="#4C72B0")
ax.bar_label(b1, labels=["隐身", "假绿", "盲"], fontsize=9)
ax.bar_label(b2, labels=["22 债入账", "leak+死登记", "cited 留痕"], fontsize=9)
ax.set_xticks(list(x)); ax.set_xticklabels(events)
ax.set_ylabel("可见的语义事件数"); ax.set_ylim(0, 25)
ax.set_title("同一事件流，两种基线策略的可见性：基线不是墙，是账本")
ax.legend()
plt.tight_layout(); plt.savefig("/root/learning-notebooks/w17d1_ledger_vs_carpet.png", dpi=110)
plt.show()

## 实验 4：docs 备份线节奏画像（读真实 git 历史）

15 个「备份」commit 的日频节奏——从「偶尔备份」长成「日频习惯」，但内容零语义（定性=无害噪声，P3 归档桶）。

In [ ]:
import subprocess, datetime
def git_docs(args):
    return subprocess.run(["git", "-C", "/root/docs", *args], capture_output=True, text=True).stdout

# 稳定窗口：9/15~9/20 的永久历史（后续 pull 不影响这段祖先提交）
out = git_logs = git_docs(["log", "--since=2026-09-15", "--until=2026-09-20",
                           "--format=%ad|%s", "--date=format:%Y-%m-%d"])
recs = [l.split("|", 1) for l in out.strip().splitlines() if l.strip()]
bak = [d for d, s in recs if s.startswith("docs: 备份")]
other = [d for d, s in recs if not s.startswith("docs: 备份")]
days = sorted({d for d, _ in recs})
bak_by_day = Counter(bak); oth_by_day = Counter(other)
print(f"窗口内 {len(recs)} commits = 备份 {len(bak)} + 非备份 {len(other)}")
for d in days:
    print(f"  {d}  备份×{bak_by_day[d]}  非备份×{oth_by_day[d]}")
assert len(bak) == 15 and bak_by_day["2026-09-17"] == 7 and bak_by_day["2026-09-16"] == 5
share = len(bak) / len(recs)
print(f"\n备份占增量 {share:.0%}——定性：无害噪声（零语义内容），但稀释受理类 commit 的可见性")

In [ ]:
# 可视化：备份线 vs 受理线日频节奏
import matplotlib.dates as mdates
xs = [datetime.date.fromisoformat(d) for d in days]
fig, ax = plt.subplots(figsize=(8.5, 4))
ax.bar(xs, [bak_by_day[d] for d in days], label="备份线（工作态快照，P3 桶）", color="#DD8452")
ax.bar(xs, [oth_by_day[d] for d in days], bottom=[bak_by_day[d] for d in days],
       label="受理/收口（digest 评级对象）", color="#4C72B0")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
ax.set_ylabel("commits/日"); ax.set_title("docs 备份线画像：日频 checkpoint 习惯（9/16×5、9/17×7 峰值）")
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig("/root/learning-notebooks/w17d1_backup_cadence.png", dpi=110)
plt.show()

## 实验 5：探针判决矩阵（probe.py 核心逻辑复现）

S2 探针从「数 commit」升级为「数表」后的判决空间：表增量 × citing 覆盖 → 判决矩阵。
citing 覆盖 100% 时增量也绿——「变化留痕，不是禁止变化」的数学表达。

In [ ]:
def probe_verdict(base, cur, changes=frozenset(), behind=0):
    if behind > 300: return "BARRIER"
    added, removed = cur - base, base - cur
    if not added and not removed: return "GREEN"
    leak = added - changes
    if removed or leak: return "RED"
    return "GREEN(cited)"

B = set(f"t{i:03d}" for i in range(499))
scen = [
    ("世界静止",           B, B, frozenset(), 0),
    ("+5 表 零 citing",    B, B | {f"n{i}" for i in range(5)}, frozenset(), 0),
    ("+5 表 全 citing",    B, B | {f"n{i}" for i in range(5)}, {f"n{i}" for i in range(5)}, 0),
    ("+22 表 零 citing（本周真实事件）", B, B | {f"n{i:02d}" for i in range(22)}, frozenset(), 55),
    ("-1 表（死登记方向）", B, B - {"t000"}, frozenset(), 0),
    ("behind 301（挡板）",  B, B, frozenset(), 301),
]
print(f"{'场景':<28s} 判决")
for name, b, c, ch, bd in scen:
    print(f"  {name:<28s} -> {probe_verdict(b, c, ch, bd)}")
assert probe_verdict(*scen[1][1:]) == "RED"
assert probe_verdict(*scen[2][1:]) == "GREEN(cited)"
assert probe_verdict(*scen[3][1:]) == "RED"      # 本周真实事件：升级前 = RED（取证入场券）
assert probe_verdict(*scen[5][1:]) == "BARRIER"
print("\n断言过：挡板>BARRIER / 集合漂移看 citing / 静止才无条件绿——fail-closed 三轴齐全")

In [ ]:
# 可视化：citing 覆盖 × 新表数 的判决矩阵
import numpy as np
covs = [0, 0.5, 1.0]; news = [0, 5, 12, 22]
code = {"GREEN": 0, "GREEN(cited)": 1, "RED": 2, "BARRIER": 3}
M = np.zeros((len(news), len(covs)))
for i, n in enumerate(news):
    for j, cv in enumerate(covs):
        if n == 0: M[i, j] = 0; continue
        added = {f"x{k}" for k in range(n)}
        cited = set(list(added)[: int(n * cv)])
        M[i, j] = code[probe_verdict(B, B | added, cited)]
fig, ax = plt.subplots(figsize=(6.8, 4.4))
im = ax.imshow(M, cmap="RdYlGn", vmin=0, vmax=2)
ax.set_xticks(range(len(covs))); ax.set_xticklabels([f"citing {c:.0%}" for c in covs])
ax.set_yticks(range(len(news))); ax.set_yticklabels([f"+{n} 表" for n in news])
for i in range(len(news)):
    for j in range(len(covs)):
        ax.text(j, i, ["绿", "绿(留痕)", "红"][int(M[i, j])], ha="center", va="center", fontweight="bold")
ax.set_title("G-07 判决矩阵：变化留痕，不是禁止变化")
plt.tight_layout(); plt.savefig("/root/learning-notebooks/w17d1_probe_matrix.png", dpi=110)
plt.show()
print("全部实验完成：5 组断言全过，4 张图存盘")